# The Mathematical Framework of Latent Regime Detection for DeFi Liquidity Provision

### A Probabilistic Approach to Dynamic Market-Making on Uniswap V3

---

## 1. Introduction: Formalizing the Problem

Consider an automated market maker (AMM) such as Uniswap V3, where a liquidity provider (LP) can concentrate capital within a custom price range. The profitability and risk of this strategy depend critically on the market’s *behaviour* – is the price moving slowly, allowing fees to accumulate, or is it volatile, exposing the LP to impermanent loss (IL) and loss-versus-rebalancing (LVR)?

This market behaviour is **not directly observable**. We postulate that it is governed by a finite set of **latent (hidden) regimes**, denoted by $Z_t \in \{1,2,\dots,K\}$. In our application, we interpret $K=3$ regimes as:
- $Z_t = 1$ (Goldilocks): low volatility, high fee capture – ideal for concentrated liquidity.
- $Z_t = 2$ (Trending): directional moves with moderate volatility.
- $Z_t = 3$ (Toxic): high volatility, adverse selection risk – dangerous for passive LPing.

We observe, however, a vector of market variables at each hourly step $t$:
$$
\mathbf{X}_t = \big[ \text{log\_return}_t,\; \text{swap\_count}_t,\; \text{total\_fees\_usd}_t,\; \text{realised\_vol}_t \big]^T.
$$

Our goal is to infer the most probable sequence of hidden regimes $Z_{1:T}$ given the observed sequence $\mathbf{X}_{1:T}$. This is a classic problem of **probabilistic inference in a dynamic system** and will be tackled using a Hidden Markov Model (HMM).

## 2. The Hidden Markov Model: A Joint Probability Model

An HMM is defined by two stochastic processes:

### 2.1 The Latent Markov Chain (Regime Dynamics)
The hidden state sequence $\{Z_t\}$ follows a first-order Markov chain with a time-homogeneous transition matrix $\mathbf{A} = (A_{ij})_{K\times K}$:
$$
P(Z_t = j \mid Z_{t-1} = i, Z_{t-2}, \dots) = P(Z_t = j \mid Z_{t-1} = i) = A_{ij},
$$
with $\sum_{j=1}^K A_{ij}=1$ for each row $i$. The initial state distribution is $\boldsymbol{\pi} = (\pi_1,\dots,\pi_K)$ where $\pi_i = P(Z_1 = i)$.

### 2.2 The Emission Process (Observations)
Given the current hidden state $Z_t$, the observation $\mathbf{X}_t$ is independent of all past and future observations and states:
$$
P(\mathbf{X}_t \mid Z_t = k, \mathbf{X}_{1:t-1}, Z_{1:t-1}) = P(\mathbf{X}_t \mid Z_t = k).
$$

### 2.3 The Joint Distribution
These two assumptions allow us to factor the full joint probability of states and observations:
$$
P(Z_{1:T}, \mathbf{X}_{1:T}) = P(Z_1) \prod_{t=2}^{T} P(Z_t \mid Z_{t-1}) \prod_{t=1}^{T} P(\mathbf{X}_t \mid Z_t).
$$
This elegant factorization is the foundation of all inference algorithms for HMMs, such as the forward–backward algorithm and Viterbi decoding.

## 3. The Gaussian Emission Assumption: A Modeling Choice

To make the model tractable, we assume that the emission distribution for each state is multivariate Gaussian with a diagonal covariance matrix (i.e., features are conditionally independent given the state):
$$
\mathbf{X}_t \mid Z_t = k \;\sim\; \mathcal{N}(\boldsymbol{\mu}_k,\; \boldsymbol{\Sigma}_k), \qquad \boldsymbol{\Sigma}_k = \operatorname{diag}(\sigma_{k,1}^2,\dots,\sigma_{k,d}^2).
$$

### 3.1 Parameter Estimation (M-Step)
Under this Gaussian assumption, the maximum likelihood estimates of the parameters, given the posterior state probabilities $\gamma_t(k) = P(Z_t = k \mid \mathbf{X}_{1:T})$, are the weighted sample moments:
$$
\hat{\boldsymbol{\mu}}_k = \frac{\sum_{t=1}^T \gamma_t(k)\,\mathbf{x}_t}{\sum_{t=1}^T \gamma_t(k)}, \qquad
\hat{\sigma}_{k,j}^2 = \frac{\sum_{t=1}^T \gamma_t(k)\,(x_{t,j} - \hat{\mu}_{k,j})^2}{\sum_{t=1}^T \gamma_t(k)}.
$$

### 3.2 Model Selection (BIC)
To choose the number of hidden states $K$, we use the Bayesian Information Criterion (BIC), which balances model fit against complexity:
$$
\text{BIC} = -2\ln\hat{L} + m\ln T,
$$
where $\hat{L}$ is the maximized likelihood of the model and $m$ is the number of free parameters. For our diagonal-covariance HMM, $m = K(K-1) + (K-1) + 2Kd$ (transition, initial, and emission parameters). The Gaussian assumption is crucial here because it defines the likelihood $\hat{L}$ and the parameter count.

## 4. Verification of Core Assumptions: Mathematical to Computational

Before trusting the model’s inferences, we must check its underlying assumptions. The table below summarizes each assumption, its mathematical implication for the HMM, and the statistical tests we performed (detailed in `00_stats.ipynb`).

| **Assumption** | **Mathematical Implication** | **How We Tested It** |
| :--- | :--- | :--- |
| **Stationarity** | For the transition matrix $\mathbf{A}$ to be constant over time, the data‑generating process should be (trend‑)stationary. Non‑stationary features (e.g., a trend in `swap_count`) would imply that regime dynamics themselves drift. | **ADF & KPSS tests:** We applied these complementary unit‑root and stationarity tests to each feature series. Joint interpretation (ADF reject + KPSS not reject) supports stationarity. |
| **Gaussian Emissions** | The M‑step formulas above are maximum likelihood only under normality. If the true distribution has heavy tails (as with `log_return`), the likelihood is misspecified and BIC becomes a heuristic, not an exact model selector. | **Jarque–Bera test & Q‑Q plots:** We tested normality of each feature, both globally and within heuristic state partitions. Q‑Q plots visually revealed deviations (e.g., heavy tails). |
| **Conditional Independence (Diagonal Covariance)** | Assuming $\boldsymbol{\Sigma}_k = \operatorname{diag}(\dots)$ means that, within a state, features are uncorrelated. If false, a diagonal model loses information and may require more states to compensate, potentially overfitting. | **Bartlett’s test of sphericity:** For each heuristic state, we tested $H_0$: correlation matrix = identity. Rejection suggests that a full‑covariance model would better capture within‑state structure. |
| **First‑Order Markov Property** | The model assumes $Z_t$ depends only on $Z_{t-1}$, i.e., $Z_t \perp Z_{t-2} \mid Z_{t-1}$. If longer memory exists (e.g., $Z_t$ depends on $Z_{t-2}$), the estimated $\mathbf{A}$ is a misspecified average of higher‑order dynamics. | **Billingsley likelihood ratio test:** Using the heuristic state sequence, we tested first‑order vs. second‑order Markov chains. A small $p$‑value indicates rejection of the first‑order hypothesis. |

These tests guide us in interpreting the model’s limitations and potential refinements (e.g., using full covariance, or even Student‑$t$ emissions).

## 5. From Model to Strategy: Interpreting the Output

Having estimated the model parameters via the Baum–Welch algorithm (an instance of Expectation‑Maximization), we can recover the most probable hidden state sequence using the **Viterbi algorithm**:
$$
\hat{Z}_{1:T} = \arg\max_{Z_{1:T}} P(Z_{1:T} \mid \mathbf{X}_{1:T}; \hat{\boldsymbol{\theta}}),
$$
where $\hat{\boldsymbol{\theta}}$ denotes the estimated parameters $\{\boldsymbol{\pi}, \mathbf{A}, \boldsymbol{\mu}_k, \boldsymbol{\Sigma}_k\}$.

Each inferred state corresponds to a distinct market condition, characterized by its mean emission vector $\boldsymbol{\mu}_k$:
- **Goldilocks** (low `realised_vol`, high `total_fees_usd`): ideal for concentrated liquidity. An LP should **narrow their range** to maximise fee capture.
- **Trending** (moderate `realised_vol`, directional `log_return`): a balanced environment; a balanced range may be appropriate.
- **Toxic** (high `realised_vol`, low fee efficiency): adverse selection dominates. The LP should **widen the range** or temporarily withdraw to mitigate impermanent loss and LVR.

Thus, the HMM transforms a stream of raw on‑chain data into a **probabilistic risk signal** that directly informs a dynamic liquidity provision strategy.

## 6. Conclusion

We have formalized the problem of latent market regime detection for a Uniswap V3 pool using a Hidden Markov Model. By specifying the random variables, the model structure, and the underlying assumptions, we have built a rigorous mathematical framework that connects classical probability theory to a modern DeFi application. The assumptions were critically examined through statistical tests, and the final decoded states provide actionable guidance for liquidity providers.

This work demonstrates how probabilistic time‑series models can be leveraged to enhance decision‑making in decentralized finance – a perfect illustration of the synergy between mathematical modelling and quantitative finance that lies at the heart of the master’s curriculum in stochastic processes and financial engineering.